In [ ]:
import os

import httpx
import openai
from langchain_core.messages import convert_to_openai_messages
from langchain_core.prompts import ChatPromptTemplate

API_KEY = os.getenv("MODEL_API_KEY_DEV")
if not API_KEY:
    raise RuntimeError(
        "Missing MODEL_API_KEY_DEV in environment. Restart the dev container after key generation/rotation."
    )

BASE_URL = os.getenv("MODEL_BASE_URL_CLEAN")
if not BASE_URL:
    raise RuntimeError(
        "Missing MODEL_BASE_URL_CLEAN in environment. Restart the dev container after env updates."
    )

CA_CERT_PATH = "/certs/.caroot/rootCA.pem"
verify_config = CA_CERT_PATH if os.path.exists(CA_CERT_PATH) else True
# Reasoning models are slower (longer output); allow a generous wall clock.
http_client = httpx.Client(verify=verify_config, timeout=120.0)

MODEL = "ollama_chat/deepseek-r1:7b"

template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are in a bad mood, reply in a short and concise manner. Make sure you express your bad mood in the reply.",
        ),
        ("user", "{input}"),
    ]
)

# LangChain `ChatOpenAI` follows the official OpenAI schema only; it does **not** preserve
# provider-specific fields such as `reasoning_content` when `base_url` points at LiteLLM/Ollama.
# Use the OpenAI Python client so `message.reasoning_content` / stream deltas stay available.
client = openai.OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    http_client=http_client,
    max_retries=0,
)

prompt_vars = {"input": "Hello, world! Who am I?"}
openai_messages = convert_to_openai_messages(template.invoke(prompt_vars).messages)


def ask_and_stream(client, messages, extra_body=None):
    completion = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        extra_body=extra_body,
    )
    msg = completion.choices[0].message
    rc = getattr(msg, "reasoning_content", None)
    if rc:
        print("--- reasoning_content ---\n", rc, sep="")
    else:
        print("(no reasoning_content on message; SDK may omit field if absent)")
    print("\n--- content ---\n", msg.content, sep="")

    print("\n--- stream ---\n")
    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        stream=True,
        extra_body=extra_body,
    )
    last_r = None
    for chunk in stream:
        ch = chunk.choices[0].delta

        r = getattr(ch, "reasoning_content", None)
        if r:
            if not last_r:
                print("REASONING: ", end="", flush=True)
            print(r, end="", flush=True)

        if ch.content:
            if last_r:
                print("ANSWER: ", end="", flush=True)
            print(ch.content, end="", flush=True)

        last_r = r

    print()


ask_and_stream(client, openai_messages, extra_body={"reasoning_effort": "high"})

NameError: name 'messages' is not defined